# 001 — Declare the Grain

A syntactically valid join can preserve every order and still invalidate an order-level measure.

This lab demonstrates how:

- four order rows become eight order-item rows
- distinct orders remain four
- revenue increases from `500` to `1,000`
- `COUNT(DISTINCT order_id)` fixes the count but not the repeated measure
- two valid corrections restore the expected result

**Prevention rule:** Before aggregating after a join, verify that every measure is unique at the resulting row grain.


## 1. Install the validated DuckDB version

This lab is pinned to DuckDB `1.5.4` so that the published results remain reproducible.


In [ ]:
%pip install -q duckdb==1.5.4 pandas

In [ ]:
import duckdb
import pandas as pd
from IPython.display import display

EXPECTED_DUCKDB_VERSION = "1.5.4"

print("DuckDB version:", duckdb.__version__)
assert duckdb.__version__ == EXPECTED_DUCKDB_VERSION, (
    f"Expected DuckDB {EXPECTED_DUCKDB_VERSION}, "
    f"but found {duckdb.__version__}"
)

con = duckdb.connect()


## 2. Create the test data

Grain contracts:

| Table | One row represents | Key |
|---|---|---|
| `customers` | one customer | `customer_id` |
| `orders` | one order | `order_id` |
| `order_items` | one item within an order | `(order_id, line_number)` |

`orders.order_total` is defined once per order.


In [ ]:
setup_sql = '''
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name VARCHAR NOT NULL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date DATE NOT NULL,
    order_total DECIMAL(10, 2) NOT NULL
);

CREATE TABLE order_items (
    order_id INTEGER NOT NULL,
    line_number INTEGER NOT NULL,
    product_name VARCHAR NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price DECIMAL(10, 2) NOT NULL,
    PRIMARY KEY (order_id, line_number)
);

INSERT INTO customers VALUES
    (1, 'Ada'),
    (2, 'Ben'),
    (3, 'Chen');

INSERT INTO orders VALUES
    (1001, 1, DATE '2026-07-01', 120.00),
    (1002, 1, DATE '2026-07-03',  80.00),
    (1003, 2, DATE '2026-07-04', 200.00),
    (1004, 3, DATE '2026-07-05', 100.00);

INSERT INTO order_items VALUES
    (1001, 1, 'Keyboard',  1, 70.00),
    (1001, 2, 'Mouse',     1, 50.00),
    (1002, 1, 'Cable',     2, 20.00),
    (1002, 2, 'Adapter',   1, 40.00),
    (1003, 1, 'Monitor',   1, 125.00),
    (1003, 2, 'Stand',     3, 25.00),
    (1004, 1, 'Headset',   1, 60.00),
    (1004, 2, 'Ear pads',  2, 20.00);
'''

con.execute(setup_sql)
print("Test data created.")


## 3. Inspect the source grains

In [ ]:
grain_inventory = con.sql('''
SELECT
    'customers' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT customer_id) AS distinct_primary_entities
FROM customers
UNION ALL
SELECT 'order_items', COUNT(*), COUNT(DISTINCT order_id)
FROM order_items
UNION ALL
SELECT 'orders', COUNT(*), COUNT(DISTINCT order_id)
FROM orders
ORDER BY table_name
''').df()

display(grain_inventory)


## 4. Establish the correct result at order grain

The input to the aggregation remains one row per order, so `order_total` appears exactly once per row.


In [ ]:
baseline = con.sql('''
SELECT
    c.customer_name,
    COUNT(*) AS order_count,
    SUM(o.order_total) AS revenue
FROM orders AS o
JOIN customers AS c USING (customer_id)
GROUP BY c.customer_name
ORDER BY c.customer_name
''').df()

display(baseline)

assert baseline["order_count"].tolist() == [2, 1, 1]
assert baseline["revenue"].astype(float).tolist() == [200.0, 200.0, 100.0]
assert float(baseline["revenue"].sum()) == 500.0


## 5. Reproduce the fan-out failure

The join moves the relation from one row per order to one row per order item.

`order_total` remains an order-grain measure, so it is repeated once per matching item.


In [ ]:
fanout = con.sql('''
SELECT
    c.customer_name,
    COUNT(*) AS joined_rows,
    SUM(o.order_total) AS reported_revenue
FROM orders AS o
LEFT JOIN order_items AS i USING (order_id)
JOIN customers AS c USING (customer_id)
GROUP BY c.customer_name
ORDER BY c.customer_name
''').df()

display(fanout)

assert fanout["joined_rows"].tolist() == [4, 2, 2]
assert fanout["reported_revenue"].astype(float).tolist() == [400.0, 400.0, 200.0]
assert float(fanout["reported_revenue"].sum()) == 1000.0


### Global comparison

In [ ]:
summary = con.sql('''
WITH base AS (
    SELECT COUNT(*) AS base_rows, SUM(order_total) AS correct_revenue
    FROM orders
),
fanout AS (
    SELECT
        COUNT(*) AS joined_rows,
        COUNT(DISTINCT o.order_id) AS distinct_orders,
        SUM(o.order_total) AS reported_revenue
    FROM orders AS o
    LEFT JOIN order_items AS i USING (order_id)
)
SELECT
    base_rows,
    joined_rows,
    distinct_orders,
    joined_rows * 1.0 / base_rows AS row_multiplier,
    correct_revenue,
    reported_revenue,
    reported_revenue / correct_revenue AS revenue_multiplier
FROM base
CROSS JOIN fanout
''').df()

display(summary)

row = summary.iloc[0]
assert int(row["base_rows"]) == 4
assert int(row["joined_rows"]) == 8
assert int(row["distinct_orders"]) == 4
assert float(row["correct_revenue"]) == 500.0
assert float(row["reported_revenue"]) == 1000.0


## 6. Inspect the multiplication at order grain

In [ ]:
multiplication = con.sql('''
SELECT
    o.order_id,
    COUNT(*) AS rows_after_join,
    MIN(o.order_total) AS order_total,
    SUM(o.order_total) AS repeated_order_total
FROM orders AS o
LEFT JOIN order_items AS i USING (order_id)
GROUP BY o.order_id
ORDER BY o.order_id
''').df()

display(multiplication)

assert multiplication["rows_after_join"].tolist() == [2, 2, 2, 2]


## 7. Show why `COUNT(DISTINCT ...)` is not a grain repair

The distinct count becomes correct, but the relation still contains one row per item and the order-level measure remains repeated.


In [ ]:
distinct_patch = con.sql('''
SELECT
    c.customer_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.order_total) AS reported_revenue
FROM orders AS o
LEFT JOIN order_items AS i USING (order_id)
JOIN customers AS c USING (customer_id)
GROUP BY c.customer_name
ORDER BY c.customer_name
''').df()

display(distinct_patch)

assert distinct_patch["order_count"].tolist() == [2, 1, 1]
assert float(distinct_patch["reported_revenue"].sum()) == 1000.0


## 8. Fix 1 — restore order grain before joining

Aggregate the detail table to one row per order before joining it to `orders`.


In [ ]:
fix_preaggregate = con.sql('''
WITH items_by_order AS (
    SELECT
        order_id,
        SUM(quantity * unit_price) AS item_total
    FROM order_items
    GROUP BY order_id
)
SELECT
    c.customer_name,
    COUNT(*) AS order_count,
    SUM(o.order_total) AS revenue
FROM orders AS o
LEFT JOIN items_by_order AS i USING (order_id)
JOIN customers AS c USING (customer_id)
GROUP BY c.customer_name
ORDER BY c.customer_name
''').df()

display(fix_preaggregate)

assert fix_preaggregate.equals(baseline)


## 9. Fix 2 — stay at item grain and use an item-grain measure

`quantity * unit_price` is defined once per order-item row.


In [ ]:
fix_item_measure = con.sql('''
SELECT
    c.customer_name,
    COUNT(*) AS item_rows,
    SUM(i.quantity * i.unit_price) AS revenue
FROM orders AS o
JOIN order_items AS i USING (order_id)
JOIN customers AS c USING (customer_id)
GROUP BY c.customer_name
ORDER BY c.customer_name
''').df()

display(fix_item_measure)

assert fix_item_measure["item_rows"].tolist() == [4, 2, 2]
assert fix_item_measure["revenue"].astype(float).tolist() == [200.0, 200.0, 100.0]
assert float(fix_item_measure["revenue"].sum()) == 500.0


### Reconcile item detail to the order header

In [ ]:
reconciliation = con.sql('''
SELECT
    o.order_id,
    o.order_total,
    SUM(i.quantity * i.unit_price) AS item_total,
    o.order_total - SUM(i.quantity * i.unit_price) AS difference
FROM orders AS o
JOIN order_items AS i USING (order_id)
GROUP BY o.order_id, o.order_total
ORDER BY o.order_id
''').df()

display(reconciliation)

assert (reconciliation["difference"].astype(float) == 0.0).all()


## 10. Run all verification checks

In [ ]:
tests = con.sql('''
WITH
base AS (
    SELECT COUNT(*) AS row_count, SUM(order_total) AS revenue
    FROM orders
),
fanout AS (
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT o.order_id) AS distinct_orders,
        SUM(o.order_total) AS revenue
    FROM orders AS o
    LEFT JOIN order_items AS i USING (order_id)
),
items_by_order AS (
    SELECT order_id, SUM(quantity * unit_price) AS item_total
    FROM order_items
    GROUP BY order_id
),
restored AS (
    SELECT COUNT(*) AS row_count, SUM(o.order_total) AS revenue
    FROM orders AS o
    LEFT JOIN items_by_order AS i USING (order_id)
),
reconciliation AS (
    SELECT MAX(ABS(o.order_total - item_total)) AS max_difference
    FROM orders AS o
    JOIN items_by_order AS i USING (order_id)
)
SELECT 'fan-out changes row count' AS test_name,
       fanout.row_count <> base.row_count AS passed
FROM base, fanout
UNION ALL
SELECT 'fan-out preserves distinct order count',
       fanout.distinct_orders = base.row_count
FROM base, fanout
UNION ALL
SELECT 'fan-out changes revenue',
       fanout.revenue <> base.revenue
FROM base, fanout
UNION ALL
SELECT 'pre-aggregation restores row count',
       restored.row_count = base.row_count
FROM base, restored
UNION ALL
SELECT 'pre-aggregation restores revenue',
       restored.revenue = base.revenue
FROM base, restored
UNION ALL
SELECT 'item detail reconciles to order header',
       reconciliation.max_difference = 0
FROM reconciliation
ORDER BY test_name
''').df()

display(tests)

assert tests["passed"].all(), "At least one verification check failed."
print("All verification checks passed.")


## Result

| Check | Before join | After item join |
|---|---:|---:|
| Rows | 4 | 8 |
| Distinct orders | 4 | 4 |
| Revenue | 500.00 | 1,000.00 |

The stable distinct-order count does not prove that the measure remained valid.

**Before aggregating after a join, verify that every measure is unique at the resulting row grain.**
